# Land Cover Classification: CNN vs Foundation Model
## EuroSAT RGB Tutorial

**Audience:** Geosciences PhD Students
**Runtime required:** GPU — go to `Runtime → Change runtime type → T4 GPU`

---

### What we're building

A land cover classifier on **EuroSAT** — 27,000 Sentinel-2 RGB patches across 10 classes — comparing two approaches:

| | CNN from scratch | Fine-tuned ViT (foundation model) |
|---|---|---|
| Prior knowledge | None | Pretrained on millions of RGB images |
| Parameters | ~1.3M | ~21M |
| Learning rate | 1e-3 | 2e-4 (lower to preserve pretrained weights) |
| Key idea | Learn everything from data | Adapt general representations to our task |

Foundation models are transforming geospatial AI. The core idea: instead of training from scratch, you start from a model that already understands visual structure, and fine-tune it for your specific task. Using RGB input here gives the ViT a direct advantage — its pretrained patch embedding was trained on 3-channel images, so no adaptation of the input layer is needed.

In [ ]:
# Run this cell first — sets up the environment on Google Colab
!git clone https://github.com/AdrianBaggstrom/ai_workshop.git
import os
os.chdir('ai_workshop/cnn_eurosat_tutorial')

!pip install -q transformers datasets torch torchvision timm scikit-learn seaborn tqdm
print('\n✅ Setup complete')

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU available')
else:
    print('❌ No GPU detected — go to Runtime → Change runtime type → T4 GPU')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import timm
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
]
NUM_CLASSES = len(CLASS_NAMES)

# RGB bands: Red (B04, idx 3), Green (B03, idx 2), Blue (B02, idx 1)
SELECTED_BANDS = [3, 2, 1]
N_BANDS        = len(SELECTED_BANDS)

# Per-band normalization statistics for RGB bands (mean and std of EuroSAT training set)
BAND_MEANS = np.array([1042.92, 1118.24, 1354.40], dtype=np.float32)  # R, G, B
BAND_STDS  = np.array([ 395.09,  333.00,  245.71], dtype=np.float32)  # R, G, B

N_EPOCHS   = 15
BATCH_SIZE = 64

---
## 1. Load EuroSAT

EuroSAT is a land use / land cover classification benchmark built from Sentinel-2 imagery:
- **27,000** geo-referenced patches, 64×64 pixels at 10m resolution
- **10 balanced classes** (~2,700 samples each)

We load the dataset via HuggingFace Datasets (`giswqs/EuroSAT_MS`) and extract the **RGB bands** (Red B04, Green B03, Blue B02), giving us standard 3-channel images.

In [ ]:
from datasets import load_dataset

print('Loading EuroSAT multispectral dataset (~800 MB, takes ~1 min)...')
raw = load_dataset('giswqs/EuroSAT_MS')
print(f'\n✅ Loaded')
print(f'Train samples: {len(raw["train"])}')
print(f'Test samples:  {len(raw["test"])}')

# Inspect one sample
sample = raw['train'][0]
img    = np.array(sample['image'], dtype=np.uint16)
print(f'\nImage shape: {img.shape}  (bands x height x width)')
print(f'Label: {sample["label"]} -> {CLASS_NAMES[sample["label"]]}')
print(f'Value range: {img.min()} – {img.max()} (uint16 reflectance)')

In [ ]:
class EuroSATDataset(Dataset):
    """PyTorch Dataset wrapping the HuggingFace EuroSAT_MS dataset.
    Extracts RGB bands and normalizes to zero mean, unit std.
    """
    def __init__(self, hf_split):
        self.data  = hf_split
        self.means = torch.tensor(BAND_MEANS).view(-1, 1, 1)
        self.stds  = torch.tensor(BAND_STDS).view(-1, 1, 1)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img = np.array(sample['image'], dtype=np.float32)  # (13, 64, 64)
        img = img[SELECTED_BANDS]                           # (3,  64, 64) — RGB
        img = torch.tensor(img)
        img = (img - self.means) / self.stds               # normalize
        return img, sample['label']

train_full = EuroSATDataset(raw['train'])
test_ds    = EuroSATDataset(raw['test'])

val_size   = int(0.15 * len(train_full))
train_size = len(train_full) - val_size
train_ds, val_ds = torch.utils.data.random_split(
    train_full, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Training:   {train_size:,} samples  ({len(train_loader)} batches)')
print(f'Validation: {val_size:,} samples')
print(f'Test:       {len(test_ds):,} samples')

---
## 2. Visualize the Data

Always look at your data before modelling. The images below are shown as true-colour RGB composites — exactly what both models receive as input.

In [ ]:
def make_rgb(img_tensor, percentile=2):
    """Build a display-ready RGB image from a normalized 3-band tensor."""
    img = img_tensor.numpy().copy()
    for i in range(3):                                     # denormalize
        img[i] = img[i] * BAND_STDS[i] + BAND_MEANS[i]
    rgb = img.transpose(1, 2, 0).astype(np.float32)       # (H, W, 3)
    lo, hi = np.percentile(rgb, [percentile, 100 - percentile])
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)

imgs, labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
fig.suptitle('Sample EuroSAT Images (RGB)', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(make_rgb(imgs[i]))
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 3. Build the Models

### Model A — CNN from scratch
Three convolutional blocks followed by a fully connected classifier. Everything is learned from the EuroSAT training data alone — no prior knowledge.

### Model B — Fine-tuned ViT (foundation model approach)
A Vision Transformer (ViT-Small) pretrained on ImageNet-21k. Because we use RGB input, the pretrained patch embedding works **directly** without any modification — the model already knows how to process 3-channel images. Only the final classification head is replaced for our 10 classes.

> **Why a lower learning rate for the ViT?** If you use the same lr as the CNN, the optimizer overshoots and destroys the pretrained weights in the first few epochs — called "catastrophic forgetting". A lower lr (2e-4 vs 1e-3) lets the model adapt gradually while preserving what it already knows.

In [ ]:
class CNN(nn.Module):
    """
    Simple CNN for RGB land cover classification.
    Three conv blocks (Conv -> ReLU -> MaxPool)
    followed by a fully connected classifier with dropout.
    """
    def __init__(self, n_bands=N_BANDS, n_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 64x64 -> 32x32
            nn.Conv2d(n_bands, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 2: 32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 3: 16x16 -> 8x8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cnn = CNN().to(DEVICE)
print(f'CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}')

In [ ]:
def build_vit(n_classes=NUM_CLASSES):
    """
    Load ViT-Small pretrained on ImageNet-21k.

    Because we use standard 3-channel RGB input, the pretrained patch
    embedding works directly — no channel adaptation required.
    This is a key advantage of RGB over multispectral input for fine-tuning:
    the model's low-level feature detectors transfer without modification.

    Only the classification head is replaced to output our 10 classes.
    """
    vit = timm.create_model(
        'vit_small_patch16_224',
        pretrained=True,
        num_classes=n_classes,
        img_size=64,
    )
    return vit

vit = build_vit().to(DEVICE)
print(f'ViT parameters: {sum(p.numel() for p in vit.parameters()):,}')
print('Pretrained on ImageNet-21k — patch embedding used directly for RGB input')

---
## 4. Train Both Models

Both models train for the same number of epochs on the same data. The only difference is the starting point: random weights vs pretrained visual representations.

In [ ]:
def train_model(model, train_loader, val_loader, n_epochs=N_EPOCHS,
               lr=1e-3, model_name='Model'):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_state = 0.0, None

    print(f'\nTraining {model_name} for {n_epochs} epochs...')

    for epoch in range(n_epochs):
        # --- Train ---
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- Validate ---
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                val_loss += criterion(out, labels).item()
                correct  += (out.argmax(1) == labels).sum().item()
                total    += labels.size(0)

        val_acc = correct / total
        history['train_loss'].append(train_loss / len(train_loader))
        history['val_loss'].append(val_loss   / len(val_loader))
        history['val_acc'].append(val_acc)

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:2d}/{n_epochs} | '
                  f'train loss: {history["train_loss"][-1]:.4f} | '
                  f'val loss: {history["val_loss"][-1]:.4f} | '
                  f'val acc: {val_acc:.4f}')

    print(f'  Best val accuracy: {best_acc:.4f}')
    model.load_state_dict(best_state)   # restore best checkpoint
    return history

In [ ]:
# Train CNN — standard learning rate, everything learned from scratch
history_cnn = train_model(cnn, train_loader, val_loader, lr=1e-3, model_name='CNN from scratch')

# Train ViT — lower learning rate to preserve pretrained weights
history_vit = train_model(vit, train_loader, val_loader, lr=2e-4, model_name='ViT fine-tuned')

# Save both models
import os
os.makedirs('models', exist_ok=True)
torch.save(cnn.state_dict(), 'models/cnn_eurosat.pt')
torch.save(vit.state_dict(), 'models/vit_eurosat.pt')
print('\nModels saved.')

---
## 5. Compare Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Comparison: CNN from Scratch vs ViT Fine-tuned',
             fontsize=13, fontweight='bold')

epochs = range(1, N_EPOCHS + 1)
styles = {
    'CNN from Scratch': {'color': '#e74c3c', 'history': history_cnn},
    'ViT Fine-tuned':   {'color': '#2980b9', 'history': history_vit},
}
for name, s in styles.items():
    h = s['history']
    axes[0].plot(epochs, h['train_loss'], '--', color=s['color'], alpha=0.5, label=f'{name} (train)')
    axes[0].plot(epochs, h['val_loss'],   '-',  color=s['color'],            label=f'{name} (val)')
    axes[1].plot(epochs, h['val_acc'],    '-',  color=s['color'],            label=name)

axes[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-Entropy Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set(title='Validation Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Evaluate on the Test Set

We report per-class F1 score alongside overall accuracy. F1 is more informative than accuracy for understanding *where* models struggle — a class with low F1 means the model is either missing many instances (low recall) or making many false predictions (low precision).

In [ ]:
def evaluate(model, loader, model_name='Model'):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = (all_preds == all_labels).mean()
    print(f'\n{"="*60}')
    print(f'  {model_name} — Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')
    print(f'{"="*60}')
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))
    return all_preds, all_labels

preds_cnn, true_labels = evaluate(cnn, test_loader, 'CNN from Scratch')
preds_vit, _           = evaluate(vit, test_loader, 'ViT Fine-tuned')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

for ax, preds, name in zip(axes,
                            [preds_cnn, preds_vit],
                            ['CNN from Scratch', 'ViT Fine-tuned']):
    cm = confusion_matrix(true_labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
                annot_kws={'size': 8})
    ax.set_title(f'{name}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 7. Where Do the Models Disagree?

The most informative comparison: patches where the CNN is wrong but the ViT is right. These reveal what the pretrained representations add — typically texture, fine structure, and subtle spectral patterns that the CNN doesn't learn from scratch in 15 epochs.

In [ ]:
# Collect all test images in order (test_loader has shuffle=False)
all_imgs = torch.cat([imgs for imgs, _ in test_loader])

def show_disagreements(preds_a, preds_b, true_labels, name_a='CNN', name_b='ViT', n=8):
    """Plot patches where model A is wrong but model B is correct."""
    idx = np.where((preds_a != true_labels) & (preds_b == true_labels))[0]
    print(f'{name_b} correct, {name_a} wrong: {len(idx)} / {len(true_labels)} test patches')

    if len(idx) == 0:
        print('No disagreements found.')
        return

    chosen = idx[:n]
    rows = (len(chosen) + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(14, rows * 3.5))
    axes = np.array(axes).flatten()
    fig.suptitle(f'Where {name_b} is right and {name_a} is wrong',
                 fontsize=12, fontweight='bold')

    for i, patch_idx in enumerate(chosen):
        axes[i].imshow(make_rgb(all_imgs[patch_idx]))
        axes[i].set_title(
            f'True: {CLASS_NAMES[true_labels[patch_idx]]}\n'
            f'{name_a}: {CLASS_NAMES[preds_a[patch_idx]]}\n'
            f'{name_b}: {CLASS_NAMES[preds_b[patch_idx]]}',
            fontsize=7
        )
        axes[i].axis('off')

    for ax in axes[len(chosen):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_disagreements(preds_cnn, preds_vit, true_labels)

---
## 8. Experiment: Label Efficiency

**The key practical advantage of foundation models** is that they need far fewer labeled samples to reach good performance — because the pretrained encoder already understands what land looks like.

Below we train a fresh ViT on only **10% of the training data** and compare it to the CNN trained on 100%. Try changing `FRACTION` to 0.05, 0.25, or 0.50 and see how the gap changes.

> For geoscientists: labeling remote sensing data is expensive (field campaigns, expert annotation). A model that performs well with 10× fewer labels has a real practical impact.

In [ ]:
FRACTION = 0.10  # try 0.05, 0.10, 0.25, 0.50

n_small      = int(FRACTION * train_size)
small_idx    = torch.randperm(train_size, generator=torch.Generator().manual_seed(42))[:n_small].tolist()
small_ds     = Subset(train_ds, small_idx)
small_loader = DataLoader(small_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f'Label efficiency experiment')
print(f'  CNN was trained on: {train_size:,} samples (100%)')
print(f'  ViT will train on:  {n_small:,} samples ({FRACTION*100:.0f}%)')

# Fresh ViT — don't reuse the already fine-tuned weights
vit_small = build_vit().to(DEVICE)
history_vit_small = train_model(
    vit_small, small_loader, val_loader,
    lr=2e-4, model_name=f'ViT ({FRACTION*100:.0f}% labels)'
)

preds_vit_small, _ = evaluate(vit_small, test_loader,
                               f'ViT fine-tuned ({FRACTION*100:.0f}% of labels)')
preds_cnn_full, _  = evaluate(cnn, test_loader,
                               'CNN from scratch (100% of labels)')

# Summary bar chart
acc_cnn = (preds_cnn_full  == true_labels).mean()
acc_vit = (preds_vit       == true_labels).mean()
acc_vit_small = (preds_vit_small == true_labels).mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    [f'CNN\n(100% labels)', f'ViT\n(100% labels)', f'ViT\n({FRACTION*100:.0f}% labels)'],
    [acc_cnn, acc_vit, acc_vit_small],
    color=['#e74c3c', '#2980b9', '#27ae60'],
    width=0.5, edgecolor='white'
)
for bar, val in zip(bars, [acc_cnn, acc_vit, acc_vit_small]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Test Accuracy')
ax.set_title('Label Efficiency: How few labels can the ViT get away with?',
             fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary & Key Takeaways

In this tutorial we compared two approaches to land cover classification from Sentinel-2 imagery:

**1. CNN from scratch** learns everything from the EuroSAT training data — spatial hierarchies, spectral patterns, class boundaries. It works well but requires all labeled samples to reach its best performance.

**2. Fine-tuned ViT** starts from a model pretrained on millions of images. It already understands texture, edges, and structure. Fine-tuning adapts these representations to our specific classes and spectral bands. The result is typically higher accuracy and, crucially, better performance with fewer labels.

**The practical implication for biodiversity research:** Annotating remote sensing data is expensive and time-consuming. Foundation models let you build accurate classifiers with a fraction of the labeled data, which can be the difference between a feasible and infeasible field campaign.

---
### Where to go next

- **[Prithvi-100M](https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M)** — A ViT pretrained specifically on Harmonized Landsat Sentinel-2 (HLS) data by NASA and IBM. Uses the same fine-tuning workflow as this tutorial but starts from geospatially-aware representations.
- **[BigEarthNet](https://bigearth.net)** — A much larger Sentinel-2 benchmark (~590k patches, 19 land cover classes, multi-label). The same approach scales directly.
- **[torchgeo](https://torchgeo.readthedocs.io)** — A PyTorch library for geospatial data with built-in datasets, transforms, and pretrained models including Prithvi.